# Agente de voz local con Transformers

Pipeline experimental del TFM:

```text
Microfono -> faster-whisper (STT) -> Transformer (LLM) -> TTS -> Altavoces
```

El LLM se ejecuta directamente con Hugging Face Transformers. Piper es el backend TTS ejecutable por defecto porque el proyecto ya incluye una voz española local. F5-TTS y Fish Speech quedan como alternativas para una fase posterior con versiones fijadas.

## 1. Preparacion

Selecciona el kernel `Python (VoiceAgents)`. Si faltan dependencias, ejecuta en una terminal con ese entorno activo:

```powershell
python -m pip install transformers accelerate sentencepiece
```

La primera ejecucion descargara el modelo desde Hugging Face. Se comienza con Qwen2.5 3B para reducir el consumo de memoria.

In [1]:
import importlib.util
import platform
import sys
import time
from pathlib import Path

print(f"Python: {sys.version.split()[0]}")
print(f"Ejecutable: {sys.executable}")
print(f"Sistema: {platform.platform()}")
for module_name, package_name in {
    "torch": "PyTorch",
    "transformers": "Transformers",
    "accelerate": "Accelerate",
    "faster_whisper": "faster-whisper",
    "sounddevice": "sounddevice",
    "soundfile": "soundfile",
}.items():
    status = "disponible" if importlib.util.find_spec(module_name) else "FALTA"
    print(f"{package_name}: {status}")

PIPER_MODEL = Path("models/piper/es_ES-davefx-medium.onnx")
print(f"Modelo Piper: {'disponible' if PIPER_MODEL.exists() else 'no encontrado'}")

Python: 3.12.10
Ejecutable: c:\Users\oitav\Documents\VIU\TFM\voice-agents\voiceagent\Scripts\python.exe
Sistema: Windows-11-10.0.26200-SP0
PyTorch: disponible
Transformers: disponible
Accelerate: disponible
faster-whisper: disponible
sounddevice: disponible
soundfile: disponible
Modelo Piper: disponible


In [2]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
MAX_NEW_TOKENS = 120
TEMPERATURE = 0.2
SYSTEM_PROMPT = "Eres un asistente de voz util. Responde en español y de forma concisa."
TTS_BACKEND = "piper"
AUDIO_DIR = Path("models/audio")
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
print(f"Modelo: {MODEL_NAME}; TTS: {TTS_BACKEND}")

Modelo: Qwen/Qwen2.5-3B-Instruct; TTS: piper


## 2. LLM Transformer

Esta es la primera prueba funcional. Si falla, revisa la instalacion, el acceso a Hugging Face o la memoria antes de continuar con audio.

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

load_started = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype="auto", device_map="auto")
model.eval()
MODEL_DEVICE = next(model.parameters()).device
print(f"Dispositivo: {MODEL_DEVICE}")
print(f"Carga: {time.perf_counter() - load_started:.2f} s")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

c:\Users\oitav\Documents\VIU\TFM\voice-agents\voiceagent\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\oitav\.cache\huggingface\hub\models--Qwen--Qwen2.5-3B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the cpu and disk.


Dispositivo: cpu
Carga: 84.74 s


In [ ]:
def generate_response(user_text: str, history: list[dict[str, str]] | None = None) -> tuple[str, float]:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages.extend(history or [])
    messages.append({"role": "user", "content": user_text})
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(MODEL_DEVICE)
    started = time.perf_counter()
    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=TEMPERATURE > 0, temperature=TEMPERATURE, pad_token_id=tokenizer.eos_token_id)
    new_tokens = outputs[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip(), time.perf_counter() - started

response, llm_seconds = generate_response("Explica en una frase que es un agente de voz local.")
print(response)
print(f"Latencia LLM: {llm_seconds:.2f} s")

In [ ]:
from faster_whisper import WhisperModel

stt_model = WhisperModel("base", device="cpu", compute_type="int8")

def transcribe_audio(audio_path: str) -> tuple[str, float]:
    started = time.perf_counter()
    segments, _ = stt_model.transcribe(audio_path, language="es", vad_filter=True)
    text = " ".join(segment.text.strip() for segment in segments).strip()
    return text, time.perf_counter() - started

print("STT preparado con faster-whisper/base.")

## 3. TTS y orquestador

Piper se ejecuta con el modelo local incluido. Para utilizar F5-TTS o Fish Speech, se debe crear un adaptador concreto para la version instalada; sus APIs y formatos de salida no son intercambiables automaticamente.

In [ ]:
import subprocess
from IPython.display import Audio, display

def synthesize_piper(text: str, output_path: str) -> tuple[str, float]:
    if not PIPER_MODEL.exists():
        raise FileNotFoundError(f"No existe el modelo Piper: {PIPER_MODEL}")
    output_file = Path(output_path).resolve()
    output_file.parent.mkdir(parents=True, exist_ok=True)
    started = time.perf_counter()
    result = subprocess.run([sys.executable, "-m", "piper", "--model", str(PIPER_MODEL), "--output_file", str(output_file)], input=text + "\n", text=True, encoding="utf-8", capture_output=True)
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip())
    return str(output_file), time.perf_counter() - started

def synthesize_audio(text: str, output_path: str) -> tuple[str, float]:
    if TTS_BACKEND == "piper":
        return synthesize_piper(text, output_path)
    raise ValueError("Backend TTS no implementado: añade un adaptador versionado para F5-TTS o Fish Speech.")

class VoiceAgent:
    def __init__(self) -> None:
        self.history: list[dict[str, str]] = []

    def process_audio(self, audio_path: str, output_path: str) -> dict[str, object]:
        text, stt_seconds = transcribe_audio(audio_path)
        if not text:
            return {"text": "", "response": "", "latency": {"stt": stt_seconds}}
        answer, llm_seconds = generate_response(text, self.history)
        self.history.extend([{"role": "user", "content": text}, {"role": "assistant", "content": answer}])
        audio_file, tts_seconds = synthesize_audio(answer, output_path)
        return {"text": text, "response": answer, "audio_path": audio_file, "latency": {"stt": stt_seconds, "llm": llm_seconds, "tts": tts_seconds, "total": stt_seconds + llm_seconds + tts_seconds}}

agent = VoiceAgent()
print("Orquestador preparado.")

In [ ]:
INPUT_AUDIO = Path("models/audio/prueba.wav")
if not INPUT_AUDIO.exists():
    raise FileNotFoundError(f"No se encuentra el audio: {INPUT_AUDIO}")
result = agent.process_audio(str(INPUT_AUDIO), str(AUDIO_DIR / "respuesta_transformers.wav"))
print(f"Texto: {result['text']}")
print(f"Respuesta: {result['response']}")
print(f"Latencias: {result['latency']}")
display(Audio(result["audio_path"]))

## 4. Evaluacion y ampliaciones

Registra el modelo exacto, versiones, dispositivo, memoria, transcripcion, respuesta y latencias. Repite las mismas frases para comparar Ollama y Transformers.

La siguiente fase puede añadir grabacion desde microfono, VAD, streaming, interrupciones y adaptadores versionados para F5-TTS y Fish Speech.